Inspect the KV Cache

GOAL: 
See EXACTLY what the KV cache contains, its shape, its memory footprint, and what happens to it as tokens are generated.

KEY INSIGHT: The KV cache is just a tuple of tensors. Nothing magical.
Each transformer layer stores a (K, V) pair. That's it.

For GPT-2 small:
  - 12 layers
  - 12 attention heads per layer
  - 64 dimensions per head
  - So each K or V tensor is: (batch, heads, seq_len, head_dim)
                             = (1, 12, seq_len, 64)

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
def load_model(m_name):
    """Load Model"""
    print(f"Loading Model {m_name}")
    tokenizer = AutoTokenizer.from_pretrained(m_name)
    model = AutoModelForCausalLM.from_pretrained(m_name)
    # Put the model in inference mode
    model.eval()
    return tokenizer, model
model_name = "Qwen/Qwen3-0.6B"
t, m = load_model(model_name)

Loading Model Qwen/Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [3]:
m

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [4]:
# prompt = """
# Article
# What is Prompt testing?


# Prompt testing is the systematic process of evaluating and validating the effectiveness of prompts used in AI interactions. This practice involves assessing how well a prompt elicits the desired response from an AI model, often through a series of controlled experiments and analyses.



# Understanding Prompt testing


# Prompt testing is a critical step in prompt engineering that ensures prompts are performing as intended and producing high-quality, relevant outputs from AI models. It combines elements of quality assurance, performance optimization, and user experience design tailored specifically for AI interactions.

# Key aspects of Prompt testing include:

# Systematic Evaluation: Methodical assessment of prompt performance against predefined criteria.
# Comparison Analysis: Testing multiple prompt variations to determine the most effective.
# Edge Case Identification: Exploring how prompts perform in unusual or extreme scenarios.
# User Simulation: Mimicking real-world usage patterns to assess prompt effectiveness.
# Iterative Refinement: Using test results to inform prompt improvements.


# Methods of Prompt testing


# A/B Testing: Comparing two or more prompt variations to determine which performs better.
# Stress Testing: Evaluating prompts under high load or challenging conditions.
# Semantic Analysis: Assessing the relevance and coherence of AI responses to prompts.
# User Feedback Collection: Gathering and analyzing user responses to prompt-generated outputs.
# Automated Testing: Using scripts or tools to run large-scale prompt tests efficiently.
# Cross-Model Testing: Evaluating prompt performance across different AI models.
# Scenario-based Testing: Creating specific use cases or scenarios to test prompt effectiveness.

# Advantages of Prompt testing

# Improved Reliability: Ensures prompts consistently produce expected results.
# Enhanced Efficiency: Identifies the most effective prompts, saving time and resources.
# Better User Satisfaction: Leads to more accurate and relevant AI responses.
# Risk Mitigation: Helps prevent potential issues or biases in AI outputs.
# Data-Driven Optimization: Provides concrete data for informed prompt refinement.


# Challenges and Considerations

# Subjectivity: Difficulty in defining objective criteria for "good" prompts in some contexts.
# Resource Intensity: Comprehensive testing can be time-consuming and computationally expensive.
# Model Specificity: Results may vary across different AI models or versions.
# Overfitting Risk: Excessive optimization for test cases may lead to reduced general performance.
# Evolving AI Capabilities: Testing strategies need to adapt as AI models improve and change.


# Best Practices for Prompt testing


# Clear Objectives: Define specific goals and success criteria for each prompt test.
# Diverse Test Sets: Use a wide range of inputs to ensure robust prompt performance.
# Controlled Environment: Maintain consistent testing conditions for accurate comparisons.
# Metrics Definition: Establish clear, measurable metrics for evaluating prompt effectiveness.
# Version Control: Keep track of different prompt versions and their test results.
# Regular Retesting: Periodically retest prompts to ensure continued effectiveness.
# User Involvement: Incorporate real user testing in addition to automated methods.
# Documentation: Maintain detailed records of test procedures, results, and insights.




# Question:
# Summarize prompt testing.
# """

In [5]:
prompt = "The goal of kv caching is "

### Inspect KV Cache working

In [6]:
def inspect_kv_cache(model, tokenizer, prompt: str):
    """
    Run a forward pass and examine KV cache in detail
    
    Each layer has it's own k,v and is not shared across layers.
    
    """
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    # Qwen3 chat formatting 
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt")
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
    num_tokens = model_inputs['input_ids'].shape
    print(f"Number of input tokens: {num_tokens}")
    
    
    # Forward pass with caching
    with torch.no_grad():
        outputs = model(
                input_ids=model_inputs["input_ids"],
                attention_mask=model_inputs["attention_mask"]
            )
    past_kv = outputs.past_key_values
    
    # Inspect KV Cache format
    
    print('KV Cache Structure')
    print(f"Type: {type(past_kv)}")
    print(f"Number of layers: {len(past_kv)}")
    print(f'First Layer Shape for keys: {past_kv.layers[0].keys.shape}, values: {past_kv.layers[0].values.shape}')
    print(f'Last Layer Shape for keys: {past_kv.layers[-1].keys.shape}, values: {past_kv.layers[-1].values.shape}')
    print(f"Total Layers: {len(past_kv.layers)} tensors (K and V), since there are {len(past_kv.layers)} decoder layers")
    
    
    # Inspect individual layers
    print('\n'*2)
    print("KV CACHE SHAPES (per layer)")
    for layer_idx in range(len(past_kv)):
        k, v = past_kv.layers[layer_idx].keys, past_kv.layers[layer_idx].values
        if layer_idx == 0:
            print(f"Layer {layer_idx:2d}: K={list(k.shape)}, V={list(v.shape)}") 
            # 1 = Batch size, 8 = KV Heads, 595 = input seq length, 128 = size per head
            # Grouped Query Attention is being used here, therefore we have 16 query heads and and 8 k/v heads. Each k/v is hared by 2 query heads.
            print(f"         (batch, num_heads, seq_len, head_dim)")
            print(f"         batch={k.shape[0]}, heads={k.shape[1]}, "
                  f"seq_len={k.shape[2]}, head_dim={k.shape[3]}")
        elif layer_idx <= 2:
            print(f"Layer {layer_idx:2d}: K={list(k.shape)}, V={list(v.shape)}")
        elif layer_idx == 3:
            print(f"  (same shape for all {len(past_kv)} layers)")
            
            
    # Memory Inspection:
    print('\n' * 2)
    print("MEMORY FOOTPRINT")
    total_bytes = 0
    for layer_idx in range(len(past_kv)):
        k, v = past_kv.layers[layer_idx].keys, past_kv.layers[layer_idx].values
        layer_bytes = k.nbytes + v.nbytes
        total_bytes += layer_bytes

    bytes_per_token = total_bytes / num_tokens[1]
    print(f"Total KV cache size: {total_bytes:,} bytes ({total_bytes / 1024:.1f} KB)")
    print(f"Per token: {bytes_per_token:,.0f} bytes ({bytes_per_token / 1024:.2f} KB)")
    print(f"For 1000 tokens: {bytes_per_token * 1000 / 1024 / 1024:.1f} MB")
    print(f"Dtype Keys: {past_kv.layers[0].keys[0].dtype}, Dtype Values: {past_kv.layers[0].values[0].dtype}")


inspect_kv_cache(m, t, prompt)

Number of input tokens: torch.Size([1, 19])
KV Cache Structure
Type: <class 'transformers.cache_utils.DynamicCache'>
Number of layers: 28
First Layer Shape for keys: torch.Size([1, 8, 19, 128]), values: torch.Size([1, 8, 19, 128])
Last Layer Shape for keys: torch.Size([1, 8, 19, 128]), values: torch.Size([1, 8, 19, 128])
Total Layers: 28 tensors (K and V), since there are 28 decoder layers



KV CACHE SHAPES (per layer)
Layer  0: K=[1, 8, 19, 128], V=[1, 8, 19, 128]
         (batch, num_heads, seq_len, head_dim)
         batch=1, heads=8, seq_len=19, head_dim=128
Layer  1: K=[1, 8, 19, 128], V=[1, 8, 19, 128]
Layer  2: K=[1, 8, 19, 128], V=[1, 8, 19, 128]
  (same shape for all 28 layers)



MEMORY FOOTPRINT
Total KV cache size: 2,179,072 bytes (2128.0 KB)
Per token: 114,688 bytes (112.00 KB)
For 1000 tokens: 109.4 MB
Dtype Keys: torch.bfloat16, Dtype Values: torch.bfloat16


In [7]:
def demonstrate_cache_growth(model, tokenizer, prompt, max_new_tokens:int):
    """
    See how the KV cache GROWS as tokens are generated.
    """
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    # Qwen3 chat formatting 
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt")
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
    num_tokens = model_inputs['input_ids'].shape
    print(f"Number of input tokens: {num_tokens}")

    print("═" * 60)
    print("KV CACHE GROWTH DURING GENERATION")
    print("═" * 60)

    past_key_values = None
    all_tokens = model_inputs['input_ids']

    with torch.no_grad():
        for step in range(max_new_tokens):
            if past_key_values is None:
                outputs = model(
                input_ids=model_inputs["input_ids"],
                attention_mask=model_inputs["attention_mask"],
                use_cache=True
            )
            else:
                outputs = model(next_token, past_key_values=past_key_values, use_cache=True)

            past_key_values = outputs.past_key_values
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            all_tokens = torch.cat([all_tokens, next_token], dim=-1)

            # Show cache size at this step
            # keys shape: (batch, heads, seq_len, head_dim) — index [2] is seq_len
            cache_seq_len = past_key_values.layers[0].keys.shape[2]
            cache_bytes = 0
            for idx in range(len(past_key_values)):
                k, v = past_key_values.layers[idx].keys, past_key_values.layers[idx].values
                layer_cache_bytes = k.nbytes + v.nbytes
                cache_bytes += layer_cache_bytes

            decoded_so_far = tokenizer.decode(all_tokens[0])
            print(f"Step {step}: cache_seq_len={cache_seq_len:3d}, "
                  f"cache={cache_bytes/1024:.1f}KB, "
                  f"text='{decoded_so_far}'")

    print()
    print("OBSERVE: cache_seq_len grows by 1 each step.")
    print("The cache accumulates K/V for every token seen so far.")

demonstrate_cache_growth(m, t, prompt, 20)

Number of input tokens: torch.Size([1, 19])
════════════════════════════════════════════════════════════
KV CACHE GROWTH DURING GENERATION
════════════════════════════════════════════════════════════
Step 0: cache_seq_len= 19, cache=2128.0KB, text='<|im_start|>user
The goal of kv caching is <|im_end|>
<|im_start|>assistant
<think>

</think>

The'
Step 1: cache_seq_len= 20, cache=2240.0KB, text='<|im_start|>user
The goal of kv caching is <|im_end|>
<|im_start|>assistant
<think>

</think>

The goal'
Step 2: cache_seq_len= 21, cache=2352.0KB, text='<|im_start|>user
The goal of kv caching is <|im_end|>
<|im_start|>assistant
<think>

</think>

The goal of'
Step 3: cache_seq_len= 22, cache=2464.0KB, text='<|im_start|>user
The goal of kv caching is <|im_end|>
<|im_start|>assistant
<think>

</think>

The goal of **'
Step 4: cache_seq_len= 23, cache=2576.0KB, text='<|im_start|>user
The goal of kv caching is <|im_end|>
<|im_start|>assistant
<think>

</think>

The goal of **KV'
Step 5: cache_seq_

In [8]:
def verify_cache_correctness(model, tokenizer):
    """
    Verify that using a cached prefix gives the
    EXACT same results as computing from scratch.

    KEY: We must split the SAME tokenized sequence — not re-wrap
    prefix/suffix as separate chat messages, which would produce
    different tokens entirely.
    """
    prefix = "The capital of France is "
    suffix = "Paris. It is a beautiful "
    full_prompt = prefix + suffix
    print(f"Full prompt: '{full_prompt}'")
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    # Run in float32 for this test.
    # The model loads in bfloat16 (only ~3 decimal digits of precision). Two different
    # computation paths — full pass vs. prefix+suffix — accumulate rounding errors
    # differently across 28 layers, producing logit differences of ~0.3 in bfloat16.
    # float32 gives ~7 decimal digits, so both paths agree to atol=1e-4.
    original_dtype = next(model.parameters()).dtype
    model = model.to(torch.float32).to(device)
    model.eval()

    # Tokenize the FULL prompt as a single chat message
    messages = [{"role": "user", "content": full_prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt")
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
    num_tokens = model_inputs['input_ids'].shape[1]

    # Split the full token sequence at a midpoint
    # Both halves come from the SAME tokenization — no re-wrapping
    prefix_len = num_tokens // 2
    prefix_input_ids = model_inputs['input_ids'][:, :prefix_len]
    prefix_attention_mask = model_inputs['attention_mask'][:, :prefix_len]
    suffix_input_ids = model_inputs['input_ids'][:, prefix_len:]

    print("CORRECTNESS VERIFICATION")
    print(f"Total tokens: {num_tokens}, split at position {prefix_len}")
    print(f"Prefix token ids: {prefix_input_ids}")
    print(f"Suffix token ids: {suffix_input_ids}")
    print()

    # NOTE: apply_chat_template wraps the prompt in Qwen3 chat tokens:
    #   <|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n
    # So the last token in the sequence is the '\n\n' after </think>, NOT the last word
    # of our prompt. The predicted next token is therefore the first word of the
    # assistant's reply (e.g. "The" as in "The capital of France...").
    print(f"Last token decoded: '{tokenizer.decode([model_inputs['input_ids'][0, -1].item()])}'")
    print(f"(The model predicts what comes after this token — the start of its reply)\n")

    with torch.no_grad():
        # Full forward pass (ground truth)
        out_full = model(input_ids=model_inputs['input_ids'], attention_mask=model_inputs['attention_mask'])
        logits_full = out_full.logits[0, -1, :]  # last position logits

        # Step 1: feed prefix tokens → build KV cache
        out_prefix = model(input_ids=prefix_input_ids, attention_mask=prefix_attention_mask, use_cache=True)
        prefix_cache = out_prefix.past_key_values

        # Step 2: feed suffix tokens using prefix cache (no re-tokenization of suffix).
        # Pass the FULL attention mask so the model knows the complete context length.
        out_suffix = model(
            input_ids=suffix_input_ids,
            past_key_values=prefix_cache,
            attention_mask=model_inputs['attention_mask'],
            use_cache=True
        )
        logits_cached = out_suffix.logits[0, -1, :]

    # Restore original dtype
    model = model.to(original_dtype)

    # Compare — use atol=1e-4 (not 1e-5).
    # torch.allclose checks: |a-b| <= atol + rtol*|b|  (rtol defaults to 1e-5)
    # For logits near zero the rtol term vanishes, so atol dominates.
    # float32 residual differences (~5e-5) require atol > 1e-5.
    max_diff = (logits_full - logits_cached).abs().max().item()
    are_same = torch.allclose(logits_full, logits_cached, atol=1e-4)

    predicted_full = torch.argmax(logits_full).item()
    predicted_cached = torch.argmax(logits_cached).item()
    print(f'Logits from full forward pass: {predicted_full}\nFrom cached forward pass: {predicted_cached}')

    print(f"Max logit difference:    {max_diff:.2e}")
    print(f"Are they close (atol=1e-4)? {are_same}")
    print(f"Same predicted next token?  {predicted_full == predicted_cached}")
    print(f"Full model predicts:   '{tokenizer.decode([predicted_full])}'")
    print(f"Cached model predicts: '{tokenizer.decode([predicted_cached])}'")
    print()

    if are_same:
        print("Cached prefix gives identical results!")
    else:
        print("Results differ")

verify_cache_correctness(m, t)

Full prompt: 'The capital of France is Paris. It is a beautiful '
CORRECTNESS VERIFICATION
Total tokens: 24, split at position 12
Prefix token ids: tensor([[151644,    872,    198,    785,   6722,    315,   9625,    374,  12095,
             13,   1084,    374]], device='mps:0')
Suffix token ids: tensor([[   264,   6233,    220, 151645,    198, 151644,  77091,    198, 151667,
            271, 151668,    271]], device='mps:0')

Last token decoded: '

'
(The model predicts what comes after this token — the start of its reply)

Logits from full forward pass: 785
From cached forward pass: 785
Max logit difference:    5.72e-05
Are they close (atol=1e-4)? True
Same predicted next token?  True
Full model predicts:   'The'
Cached model predicts: 'The'

Cached prefix gives identical results!
